In [4]:
import requests
import os
import json

In [2]:
TIMEOUT = 10

In [3]:
if not os.path.exists("../matches/match_ids.json"):
    with open("../matches/match_ids.json", "w") as f:
        json.dump([], f)

In [4]:
def get_avg_rank(players: list[dict[str, str | int]]) -> int:
    total_rank: int = 0
    for player in players:
        player_rank: int = player.get("rank_tier", 0)
        total_rank += player_rank if player_rank is not None else 0
    return total_rank // 10

In [5]:
def get_match_data(match_id: int) -> dict[str, str | int] | None:
    try:
        response: requests.Response = requests.get(f"https://api.opendota.com/api/matches/{match_id}", timeout=(5, 30))
        response.raise_for_status()
        data = response.json()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching match ID {match_id}: {e}")
        return None
    
    if data["game_mode"] in {1, 2, 22}:
        return {"match_id": data["match_id"], "replay_url": data["replay_url"], "avg_rank": get_avg_rank(data["players"])}
    return None

In [6]:
while True:
    saved_matches: list[dict[str, str | int]] = []

    try:
        with open("../matches/match_ids.jsonl", "r") as f:
            for line in f:
                saved_matches.append(json.loads(line))
    except FileNotFoundError:
        pass

    min_match_id: int = (
        min(match["match_id"] for match in saved_matches)
        if saved_matches
        else 9000000000 
    )

    if len(saved_matches) >= 2000:
        break

    try:
        response: requests.Response = requests.get(
            f"https://api.opendota.com/api/parsedMatches?less_than_match_id={min_match_id}",
            timeout=(5, 30)
        )
        response.raise_for_status()
        data: list[dict] = response.json()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching match IDs: {e}")
        continue

    saved_matches_ids: set[int] = {
        match["match_id"] for match in saved_matches
    }

    new_match_ids: list[int] = [
        match["match_id"]
        for match in data
        if match["match_id"] not in saved_matches_ids
    ]

    print(f"already collected match_ids:{len(saved_matches_ids)}")
    print(new_match_ids)

    with open("../matches/match_ids.jsonl", "a") as f:
        for i in new_match_ids:
            match_replay: dict[int, str] | None = get_match_data(i)

            if match_replay is not None:
                f.write(json.dumps(match_replay) + "\n")

already collected match_ids:0
[8803564352, 8803563960, 8803558580, 8803557962, 8803551997, 8803551150, 8803550505, 8803549626, 8803549013, 8803546883, 8803546611, 8803545896, 8803545873, 8803545795, 8803545526, 8803545198, 8803544076, 8803543254, 8803542709, 8803542358, 8803542357, 8803542297, 8803542115, 8803541894, 8803541653, 8803541011, 8803540953, 8803540251, 8803540095, 8803539067, 8803538994, 8803538961, 8803538859, 8803538725, 8803538620, 8803538538, 8803538434, 8803538354, 8803537928, 8803537288, 8803537004, 8803536351, 8803536303, 8803536020, 8803535600, 8803535564, 8803535459, 8803535202, 8803534910, 8803534821, 8803534640, 8803534015, 8803533794, 8803532819, 8803532735, 8803532443, 8803532133, 8803531532, 8803531484, 8803531303, 8803530421, 8803530119, 8803529453, 8803529388, 8803529287, 8803528879, 8803528432, 8803528356, 8803528342, 8803528199, 8803527908, 8803527831, 8803527632, 8803527630, 8803527610, 8803527516, 8803527382, 8803527311, 8803527140, 8803526637, 880352652

In [8]:
json_path = "../matches/match_ids.json"
jsonl_path = "../matches/match_ids.jsonl"

# читаем обычный json
with open(json_path, "r") as f:
    data: list[dict] = json.load(f)

# читаем jsonl
with open(jsonl_path, "r") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue

        data.append(json.loads(line))

# перезаписываем json
with open(json_path, "w") as f:
    json.dump(data, f, indent=4)

In [9]:
from statistics import mean, stdev, median
with open("../matches/match_ids.json", "r") as f:
    saved_matches: list[dict[str, str | int]] = json.load(f)
    avg_ranks: list[int] = [match["avg_rank"] for match in saved_matches if "avg_rank" in match]
    print(f"Average rank: {mean(avg_ranks)}")
    print(f"Standard deviation: {stdev(avg_ranks)}")
    print(f"Median: {median(avg_ranks)}")
    print(f"Len: {len(avg_ranks)}") # 1027

Average rank: 54.17210682492582
Standard deviation: 22.675671295021843
Median: 56
Len: 3033


In [1]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import json
import requests

with open("../matches/match_ids.json", "r") as f:
    saved_matches = json.load(f)

ALL_URLS = {match["match_id"]: match["replay_url"] for match in saved_matches if "replay_url" in match}
FINISHED_URLS = {int(file.stem) for file in Path("../matches/parses").iterdir() if file.is_file()}

URLS = {str(match_id) : url for match_id, url in ALL_URLS.items() if match_id not in FINISHED_URLS}

OUTPUT_DIR = Path("../matches/replays")
OUTPUT_DIR.mkdir(exist_ok=True)

TIMEOUT = (5, 120)
MAX_WORKERS = 14

def download(url: str, match_id: str):
    filename = f"{match_id}.dem.bz2"
    path = OUTPUT_DIR / filename

    try:
        with requests.get(url, stream=True, timeout=TIMEOUT) as r:
            r.raise_for_status()

            with open(path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                    if chunk:
                        f.write(chunk)

        return f"OK: {filename}"

    except Exception as e:
        return f"FAIL: {filename} -> {e}"


with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(download, url, match_id) for match_id, url in URLS.items()]

    for future in as_completed(futures):
        print(future.result())

OK: 8803563960.dem.bz2
OK: 8803542115.dem.bz2
OK: 8803542358.dem.bz2
OK: 8803544076.dem.bz2
OK: 8803538961.dem.bz2
OK: 8803538538.dem.bz2
OK: 8803542709.dem.bz2
OK: 8803542357.dem.bz2
OK: 8803540953.dem.bz2
OK: 8803540095.dem.bz2
OK: 8803539067.dem.bz2
OK: 8803545795.dem.bz2
OK: 8803543254.dem.bz2
OK: 8803535564.dem.bz2
OK: 8803535202.dem.bz2
OK: 8803536303.dem.bz2
OK: 8803534015.dem.bz2
OK: 8803534910.dem.bz2
OK: 8803537004.dem.bz2
OK: 8803538434.dem.bz2
OK: 8803540251.dem.bz2
OK: 8803528342.dem.bz2
OK: 8803535459.dem.bz2
OK: 8803532735.dem.bz2
OK: 8803527630.dem.bz2
OK: 8803531484.dem.bz2
OK: 8803534640.dem.bz2
OK: 8803529453.dem.bz2
OK: 8803526637.dem.bz2
OK: 8803538620.dem.bz2
OK: 8803527908.dem.bz2
OK: 8803526291.dem.bz2
OK: 8803527632.dem.bz2
OK: 8803527311.dem.bz2
OK: 8803545896.dem.bz2
OK: 8803525885.dem.bz2
OK: 8803533794.dem.bz2
OK: 8803526460.dem.bz2
OK: 8803526526.dem.bz2
OK: 8803525992.dem.bz2
OK: 8803525423.dem.bz2
OK: 8803525707.dem.bz2
OK: 8803527516.dem.bz2
OK: 8803525

In [2]:
from pathlib import Path
import bz2
import shutil

INPUT_DIR = Path("../matches/replays")
OUTPUT_DIR = Path("../matches/replays_dem")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for bz2_file in INPUT_DIR.iterdir():
    if not bz2_file.is_file():
        continue

    if not bz2_file.name.endswith(".bz2"):
        continue

    # убираем .bz2 → получаем .dem
    output_file = OUTPUT_DIR / bz2_file.with_suffix("").name

    if output_file.exists():
        print(f"SKIP (already exists): {output_file.name}")
        continue

    try:
        with bz2.open(bz2_file, "rb") as src:
            with open(output_file, "wb") as dst:
                shutil.copyfileobj(src, dst)

        print(f"OK: {bz2_file.name} -> {output_file.name}")

    except Exception as e:
        print(f"FAIL: {bz2_file.name} -> {e}")

FAIL: 8802810080.dem.bz2 -> Compressed file ended before the end-of-stream marker was reached
OK: 8803349660.dem.bz2 -> 8803349660.dem
OK: 8802569160.dem.bz2 -> 8802569160.dem
OK: 8803515466.dem.bz2 -> 8803515466.dem
FAIL: 8802994330.dem.bz2 -> Compressed file ended before the end-of-stream marker was reached
OK: 8803470630.dem.bz2 -> 8803470630.dem
OK: 8803464494.dem.bz2 -> 8803464494.dem
OK: 8803175498.dem.bz2 -> 8803175498.dem
FAIL: 8803146488.dem.bz2 -> Compressed file ended before the end-of-stream marker was reached
OK: 8803158613.dem.bz2 -> 8803158613.dem
FAIL: 8802543635.dem.bz2 -> Compressed file ended before the end-of-stream marker was reached
FAIL: 8803418799.dem.bz2 -> Compressed file ended before the end-of-stream marker was reached
OK: 8803232461.dem.bz2 -> 8803232461.dem
FAIL: 8802567665.dem.bz2 -> Compressed file ended before the end-of-stream marker was reached
OK: 8803324136.dem.bz2 -> 8803324136.dem
OK: 8802925688.dem.bz2 -> 8802925688.dem
OK: 8803458788.dem.bz2 -> 

In [3]:
from pathlib import Path
import requests

INPUT_DIR = Path("../matches/replays_dem")
OUTPUT_DIR = Path("../matches/parses")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TIMEOUT = (5, 300)

for dem_file in INPUT_DIR.iterdir():
    if not dem_file.is_file():
        continue

    if not dem_file.name.endswith(".dem"):
        continue

    output_file = OUTPUT_DIR / f"{dem_file.stem}.jsonl"

    # если уже обработан — пропускаем
    if output_file.exists():
        print(f"SKIP: {dem_file.name}")
        continue

    try:
        with open(dem_file, "rb") as f:
            response = requests.post(
                "http://localhost:5600",
                data=f,
                headers={
                    "Content-Type": "application/octet-stream"
                },
                timeout=TIMEOUT
            )

        response.raise_for_status()

        # сохраняем jsonl как есть
        with open(output_file, "w") as out:
            out.write(response.text)

        print(f"OK: {dem_file.name} -> {output_file.name}")

    except Exception as e:
        print(f"FAIL: {dem_file.name} -> {e}")

OK: 8802546074.dem -> 8802546074.jsonl
OK: 8802712474.dem -> 8802712474.jsonl
OK: 8802717589.dem -> 8802717589.jsonl
OK: 8803207092.dem -> 8803207092.jsonl
OK: 8803173367.dem -> 8803173367.jsonl
OK: 8803534821.dem -> 8803534821.jsonl
OK: 8803266892.dem -> 8803266892.jsonl
OK: 8802503599.dem -> 8802503599.jsonl
OK: 8803334026.dem -> 8803334026.jsonl
OK: 8803495288.dem -> 8803495288.jsonl
OK: 8803356463.dem -> 8803356463.jsonl
OK: 8802548540.dem -> 8802548540.jsonl
OK: 8803489831.dem -> 8803489831.jsonl
OK: 8802892505.dem -> 8802892505.jsonl
OK: 8803231212.dem -> 8803231212.jsonl
OK: 8803391538.dem -> 8803391538.jsonl
OK: 8803352437.dem -> 8803352437.jsonl
OK: 8802586604.dem -> 8802586604.jsonl
OK: 8803199782.dem -> 8803199782.jsonl
OK: 8802933553.dem -> 8802933553.jsonl
OK: 8802543091.dem -> 8802543091.jsonl
OK: 8803338259.dem -> 8803338259.jsonl
OK: 8802580705.dem -> 8802580705.jsonl
OK: 8803470079.dem -> 8803470079.jsonl
OK: 8802551185.dem -> 8802551185.jsonl
OK: 8802859192.dem -> 880

In [8]:
!ls ../matches/parses | wc -l

2617
